# Balanced Interleaved DQN Fine-Tuning Notebook (Kaggle)

Notebook này fine-tune từ model `dqn_eval_best.pt` đã train theo balanced reward. Mục tiêu là cân bằng nhiều tình huống giao thông thay vì sửa riêng một hướng rồi làm hỏng hướng còn lại.

Workflow mới:

1. Tải base checkpoint từ Hugging Face.
2. Tạo nhiều scenario SUMO: normal, heavy, EW peak, NS peak, mixed, và các biến thể scale.
3. Train DQN theo `scenario sampler`: mỗi episode chọn ngẫu nhiên một scenario theo trọng số.
4. Reward vẫn phạt total wait, max phase wait, queue imbalance, max vehicle wait.
5. Chọn checkpoint bằng hard guardrail theo từng seed, không chỉ điểm trung bình.

Output chính vẫn là `dqn_ew_repair_best.pt`, nhưng chỉ được promote nếu pass hard guardrail. Nếu không pass, notebook giữ lại base `dqn_eval_best.pt` để tránh deploy model overfit.


## 1. Install dependencies and clone `dev-truong`

In [ ]:
!pip install -q eclipse-sumo traci sumolib libsumo
!pip install -q pydantic pydantic-settings structlog gymnasium numpy tqdm huggingface_hub

import os
from pathlib import Path
import subprocess

repo_dir = Path("/kaggle/working/smart-traffic-light-system")
if not repo_dir.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "dev-truong",
            "--single-branch",
            "https://github.com/truongNgn/smart-traffic-light-system.git",
            str(repo_dir),
        ],
        check=True,
    )

os.chdir(repo_dir)
print("cwd =", Path.cwd())


## 2. Configure SUMO and verify the 2-phase action space

In [ ]:
import os
import sumo

os.environ["SUMO_HOME"] = os.path.dirname(sumo.__file__)
print("SUMO_HOME =", os.environ["SUMO_HOME"])

!sumo --version

from common.constants import DQN_OUTPUT_SIZE, NUM_ACTIONS, PHASE_DIRECTIONS, PhaseAction
from rl.env.traffic_env import SumoTrafficEnv

assert NUM_ACTIONS == 2
assert DQN_OUTPUT_SIZE == 2
assert set(PhaseAction) == {PhaseAction.EAST_WEST, PhaseAction.NORTH_SOUTH}
assert len(PHASE_DIRECTIONS[PhaseAction.EAST_WEST]) == 2
assert len(PHASE_DIRECTIONS[PhaseAction.NORTH_SOUTH]) == 2

print("Verified 2-phase action space:", PHASE_DIRECTIONS)


## 3. Download the trained base checkpoint from Hugging Face

Mặc định notebook tải `dqn_eval_best.pt` từ Hugging Face repo:

```text
truongNgn/smart-traffic-light-dqn-sumo
```

Nếu HF repo của bạn public, chỉ cần bật Internet trong Kaggle là chạy được. Nếu repo private, tạo Kaggle Secret tên `HF_TOKEN` rồi cell sẽ tự dùng token đó. Vẫn có fallback: nếu không tải được từ HF, notebook sẽ tìm checkpoint trong `/kaggle/input` hoặc `/kaggle/working/checkpoints`.


In [ ]:
from pathlib import Path
import shutil
import torch
from huggingface_hub import hf_hub_download

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Sửa 2 biến này nếu bạn đặt Hugging Face repo/file khác.
HF_REPO_ID = "johnnynnt/smart-traffic-light-dqn-sumo"
HF_FILENAME = "dqn_eval_best.pt"
HF_REPO_TYPE = "model"

# Optional fallback: nếu muốn chỉ định file local/Kaggle input thủ công.
INPUT_CHECKPOINT_PATH = None


def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("HF_TOKEN")
        return token or None
    except Exception:
        return None


def find_fallback_checkpoint():
    if INPUT_CHECKPOINT_PATH:
        path = Path(INPUT_CHECKPOINT_PATH)
        if path.exists():
            return path
        raise FileNotFoundError(f"INPUT_CHECKPOINT_PATH không tồn tại: {path}")

    candidates = [
        CHECKPOINT_DIR / "dqn_eval_best.pt",
        *Path("/kaggle/input").glob("**/dqn_eval_best.pt"),
        *Path("/kaggle/input").glob("**/dqn_best.pt"),
        *Path("/kaggle/input").glob("**/dqn_final.pt"),
    ]
    candidates = [p for p in candidates if p.exists()]
    return candidates[0] if candidates else None


target_checkpoint = CHECKPOINT_DIR / "dqn_eval_best.pt"

try:
    hf_path = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=HF_FILENAME,
        repo_type=HF_REPO_TYPE,
        token=get_hf_token(),
    )
    shutil.copy2(hf_path, target_checkpoint)
    print("Downloaded from Hugging Face:", HF_REPO_ID, HF_FILENAME)
    print("HF cache path:", hf_path)
except Exception as exc:
    print("HF download failed, fallback to Kaggle/local checkpoint.")
    print(type(exc).__name__, exc)
    fallback = find_fallback_checkpoint()
    if fallback is None:
        raise FileNotFoundError(
            "Không tìm thấy checkpoint. Hãy upload lên Hugging Face, hoặc add Kaggle Dataset chứa dqn_eval_best.pt."
        )
    if fallback.resolve() != target_checkpoint.resolve():
        shutil.copy2(fallback, target_checkpoint)
    print("Using fallback checkpoint:", fallback)

data = torch.load(target_checkpoint, map_location="cpu")
state_dict = data.get("policy_state_dict") or data.get("model_state_dict") or data
last_shapes = [(k, tuple(v.shape)) for k, v in state_dict.items() if hasattr(v, "shape")][-6:]

print("Base checkpoint ready:", target_checkpoint)
print("Episode:", data.get("episode") if isinstance(data, dict) else None)
print("Last layer hints:", last_shapes)

assert last_shapes[-1][1] == (2,), "Checkpoint này không phải model 2-action."


## 4. Build scenario bank for interleaved training

Thay vì train tuần tự `normal -> heavy -> EW -> NS`, notebook tạo một ngân hàng scenario. Mỗi episode sẽ sample một scenario khác nhau, giúp replay buffer chứa kinh nghiệm đa dạng và giảm hiện tượng model quên case trước đó.

Trọng số train mục tiêu:

- Normal: khoảng 24%
- Heavy balanced: khoảng 23%
- EW peak: khoảng 20%
- NS peak: khoảng 20%
- Mixed: khoảng 13%


In [ ]:
!python -m simulation.net.build_net
!python -m simulation.net.generate_routes --duration 1800 --seed 42 --out simulation/net/intersection.rou.xml

from copy import deepcopy
from pathlib import Path
import xml.etree.ElementTree as ET


def flow_origin(flow):
    route = flow.get("route", "")
    if route.startswith("route_"):
        return route.split("_")[1]
    parts = flow.get("id", "").split("_")
    return parts[-2] if len(parts) >= 2 else ""


def write_sumocfg(path, route_file):
    path.write_text(f"""<?xml version="1.0" encoding="UTF-8"?>
<configuration>
    <input>
        <net-file value="intersection.net.xml"/>
        <route-files value="{route_file}"/>
        <additional-files value="vtypes.add.xml"/>
    </input>
    <time>
        <begin value="0"/>
        <step-length value="1"/>
    </time>
    <processing>
        <time-to-teleport value="-1"/>
    </processing>
    <report>
        <no-step-log value="true"/>
        <duration-log.disable value="true"/>
    </report>
</configuration>
""", encoding="utf-8")


def scaled_rate(flow, *, all_scale=1.0, ew_scale=1.0, ns_scale=1.0):
    origin = flow_origin(flow)
    axis_scale = ew_scale if origin in {"E", "W"} else ns_scale
    return float(flow.get("vehsPerHour")) * all_scale * axis_scale


def make_scaled_scenario(name, *, duration_s=1800, all_scale=1.0, ew_scale=1.0, ns_scale=1.0):
    src = Path("simulation/net/intersection.rou.xml")
    route_path = Path(f"simulation/net/{name}.rou.xml")
    sumocfg_path = Path(f"simulation/net/{name}.sumocfg")
    tree = ET.parse(src)
    root = tree.getroot()

    for flow in root.findall("flow"):
        flow.set("end", str(duration_s))
        flow.set("vehsPerHour", f"{scaled_rate(flow, all_scale=all_scale, ew_scale=ew_scale, ns_scale=ns_scale):.2f}")

    ET.indent(tree, space="    ")
    tree.write(route_path, encoding="UTF-8", xml_declaration=True)
    write_sumocfg(sumocfg_path, route_path.name)
    return str(sumocfg_path)


def make_time_block_scenario(name, blocks):
    src = Path("simulation/net/intersection.rou.xml")
    route_path = Path(f"simulation/net/{name}.rou.xml")
    sumocfg_path = Path(f"simulation/net/{name}.sumocfg")
    base_tree = ET.parse(src)
    base_root = base_tree.getroot()

    root = ET.Element(base_root.tag, base_root.attrib)
    for route in base_root.findall("route"):
        root.append(deepcopy(route))

    for block_name, begin, end, multipliers in blocks:
        for flow in base_root.findall("flow"):
            block_flow = deepcopy(flow)
            block_flow.set("id", f"{flow.get('id')}_{block_name}")
            block_flow.set("begin", str(begin))
            block_flow.set("end", str(end))
            block_flow.set("vehsPerHour", f"{scaled_rate(flow, **multipliers):.2f}")
            root.append(block_flow)

    tree = ET.ElementTree(root)
    ET.indent(tree, space="    ")
    tree.write(route_path, encoding="UTF-8", xml_declaration=True)
    write_sumocfg(sumocfg_path, route_path.name)
    return str(sumocfg_path)


SCENARIOS = {
    "normal": "simulation/net/intersection.sumocfg",
    "heavy_1_5x": make_scaled_scenario("intersection_heavy_1_5x", all_scale=1.5),
    "heavy_2x": make_scaled_scenario("intersection_heavy_2x", all_scale=2.0),
    "heavy_2_5x": make_scaled_scenario("intersection_heavy_2_5x", all_scale=2.5),
    "imbalanced_ew2x": make_scaled_scenario("intersection_imbalanced_ew2x", ew_scale=2.0),
    "imbalanced_ew3x": make_scaled_scenario("intersection_imbalanced_ew3x", ew_scale=3.0),
    "imbalanced_ew3_5x": make_scaled_scenario("intersection_imbalanced_ew3_5x", ew_scale=3.5),
    "imbalanced_ns2x": make_scaled_scenario("intersection_imbalanced_ns2x", ns_scale=2.0),
    "imbalanced_ns3x": make_scaled_scenario("intersection_imbalanced_ns3x", ns_scale=3.0),
    "imbalanced_ns3_5x": make_scaled_scenario("intersection_imbalanced_ns3_5x", ns_scale=3.5),
}

SCENARIOS["mixed_balanced_peak"] = make_time_block_scenario(
    "intersection_mixed_balanced_peak",
    [
        ("normal", 0, 600, dict(all_scale=1.0)),
        ("heavy2x", 600, 1200, dict(all_scale=2.0)),
        ("ew3x", 1200, 1800, dict(ew_scale=3.0)),
        ("ns3x", 1800, 2400, dict(ns_scale=3.0)),
    ],
)

SCENARIOS["mixed_extreme_peak"] = make_time_block_scenario(
    "intersection_mixed_extreme_peak",
    [
        ("normal", 0, 400, dict(all_scale=1.0)),
        ("heavy25x", 400, 900, dict(all_scale=2.5)),
        ("ew35x", 900, 1500, dict(ew_scale=3.5)),
        ("ns35x", 1500, 2100, dict(ns_scale=3.5)),
        ("normal_tail", 2100, 2400, dict(all_scale=1.0)),
    ],
)

SCENARIO_DURATIONS = {
    "normal": 1800,
    "heavy_1_5x": 1800,
    "heavy_2x": 1800,
    "heavy_2_5x": 1800,
    "imbalanced_ew2x": 1800,
    "imbalanced_ew3x": 1800,
    "imbalanced_ew3_5x": 1800,
    "imbalanced_ns2x": 1800,
    "imbalanced_ns3x": 1800,
    "imbalanced_ns3_5x": 1800,
    "mixed_balanced_peak": 2400,
    "mixed_extreme_peak": 2400,
}

# Full benchmark durations stay realistic. Training uses shorter windows so Kaggle
# does not spend most of its time simulating long jam tails after the learning
# signal has already appeared.
TRAIN_SCENARIO_DURATIONS = {name: min(duration_s, 1200) for name, duration_s in SCENARIO_DURATIONS.items()}
TRAIN_SCENARIO_DURATIONS["mixed_balanced_peak"] = 1600
TRAIN_SCENARIO_DURATIONS["mixed_extreme_peak"] = 1600

TRAIN_SCENARIO_WEIGHTS = {
    "normal": 0.24,
    "heavy_1_5x": 0.08,
    "heavy_2x": 0.12,
    "heavy_2_5x": 0.03,
    "imbalanced_ew2x": 0.08,
    "imbalanced_ew3x": 0.10,
    "imbalanced_ew3_5x": 0.02,
    "imbalanced_ns2x": 0.08,
    "imbalanced_ns3x": 0.10,
    "imbalanced_ns3_5x": 0.02,
    "mixed_balanced_peak": 0.10,
    "mixed_extreme_peak": 0.03,
}

assert abs(sum(TRAIN_SCENARIO_WEIGHTS.values()) - 1.0) < 1e-9
SCENARIOS


## 5. Patch reward for EW repair without EW hard-bias

In [ ]:
from pathlib import Path
import importlib

reward_path = Path("rl/reward/waiting_time_reward.py")
reward_path.write_text(r'''"""Balanced repair reward for traffic-police-style DQN fine-tuning."""

from __future__ import annotations

from common.constants import PHASE_DIRECTIONS, STOPPED_SPEED_THRESHOLD_MPS
from simulation.state.grid_encoder import APPROACH_EDGE_BY_DIRECTION
from simulation.state.waiting_time import total_waiting_time


class WaitingTimeReward:
    def __init__(
        self,
        *,
        total_wait_delta_weight: float = 1.0,
        total_wait_level_weight: float = 0.035,
        max_phase_wait_weight: float = 0.16,
        phase_imbalance_weight: float = 0.12,
        queue_weight: float = 3.2,
        max_phase_queue_weight: float = 7.0,
        max_vehicle_wait_weight: float = 0.12,
    ) -> None:
        self.total_wait_delta_weight = total_wait_delta_weight
        self.total_wait_level_weight = total_wait_level_weight
        self.max_phase_wait_weight = max_phase_wait_weight
        self.phase_imbalance_weight = phase_imbalance_weight
        self.queue_weight = queue_weight
        self.max_phase_queue_weight = max_phase_queue_weight
        self.max_vehicle_wait_weight = max_vehicle_wait_weight
        self._previous_total = 0.0

    def reset(self, traci_conn) -> float:  # noqa: ANN001
        self._previous_total = total_waiting_time(traci_conn)
        return self._previous_total

    def step(self, traci_conn) -> float:  # noqa: ANN001
        current_total = total_waiting_time(traci_conn)
        delta_reward = self._previous_total - current_total
        phase_waits, phase_queues, max_vehicle_wait = self._phase_metrics(traci_conn)

        wait_values = list(phase_waits.values())
        queue_values = list(phase_queues.values())
        max_phase_wait = max(wait_values) if wait_values else 0.0
        phase_imbalance = max(wait_values) - min(wait_values) if len(wait_values) >= 2 else 0.0
        total_queue = sum(queue_values)
        max_phase_queue = max(queue_values) if queue_values else 0

        reward = (
            self.total_wait_delta_weight * delta_reward
            - self.total_wait_level_weight * current_total
            - self.max_phase_wait_weight * max_phase_wait
            - self.phase_imbalance_weight * phase_imbalance
            - self.queue_weight * total_queue
            - self.max_phase_queue_weight * max_phase_queue
            - self.max_vehicle_wait_weight * max_vehicle_wait
        )
        self._previous_total = current_total
        return reward

    @staticmethod
    def _phase_metrics(traci_conn):  # noqa: ANN001
        phase_edges = {
            phase.name: {APPROACH_EDGE_BY_DIRECTION[direction] for direction in directions}
            for phase, directions in PHASE_DIRECTIONS.items()
        }
        waits = {phase_name: 0.0 for phase_name in phase_edges}
        queues = {phase_name: 0 for phase_name in phase_edges}
        max_vehicle_wait = 0.0

        for vehicle_id in traci_conn.vehicle.getIDList():
            road_id = traci_conn.vehicle.getRoadID(vehicle_id)
            waiting_time = traci_conn.vehicle.getWaitingTime(vehicle_id)
            speed = traci_conn.vehicle.getSpeed(vehicle_id)
            for phase_name, edges in phase_edges.items():
                if road_id in edges:
                    waits[phase_name] += waiting_time
                    if speed < STOPPED_SPEED_THRESHOLD_MPS:
                        queues[phase_name] += 1
                    max_vehicle_wait = max(max_vehicle_wait, waiting_time)
                    break

        return waits, queues, max_vehicle_wait
''', encoding="utf-8")

import rl.reward.waiting_time_reward as reward_module
importlib.reload(reward_module)
from rl.reward.waiting_time_reward import WaitingTimeReward

assert hasattr(WaitingTimeReward(), "_phase_metrics")
print("Repair reward patched:", reward_path)


## 6. Interleaved fine-tuning from the balanced model

Đây là phần thay đổi quan trọng nhất. Notebook không train theo stage tuần tự nữa. Thay vào đó, mỗi episode:

1. Sample một scenario từ `TRAIN_SCENARIO_WEIGHTS`.
2. Reset SUMO với seed khác nhau.
3. Cho DQN chạy episode đó.
4. Đưa transition vào replay buffer chung.
5. Train DQN từ replay buffer chung, nên kinh nghiệm normal/heavy/EW/NS/mixed được trộn lẫn.

Cách này giảm rủi ro overfit EW rồi phá NS như checkpoint repair episode 2675.


In [ ]:
import random
from collections import defaultdict, deque
from pathlib import Path

from tqdm.auto import tqdm

from benchmark.metrics import EpisodeMetrics
from rl.agent.dqn_agent import DQNAgent
from rl.agent.replay_buffer import ReplayBuffer, Transition
from rl.env.traffic_env import SumoTrafficEnv
from rl.train.checkpoint import load_checkpoint, save_checkpoint

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
BASE_CHECKPOINT = CHECKPOINT_DIR / "dqn_eval_best.pt"

GUARD_CONFIG = dict(
    max_red_time_s=None,
    soft_red_time_s=70.0,
    hard_red_time_s=120.0,
    starving_queue_threshold=4,
    starving_wait_time_s=160.0,
)

FINETUNE_EPISODES = 450
RANDOM_SEED = 20260728
LEARNING_RATE = 1.5e-5
GAMMA = 0.99
REPLAY_CAPACITY = 100_000
MIN_REPLAY_SIZE = 2_000
BATCH_SIZE = 128
TRAIN_EVERY_N_STEPS = 4
TARGET_SYNC_EVERY_EPISODES = 5
CHECKPOINT_EVERY_EPISODES = 75
EPSILON_START = 0.14
EPSILON_END = 0.02
EPSILON_DECAY_EPISODES = 350


def resolve_backend():
    try:
        import libsumo  # noqa: F401

        return "libsumo"
    except ImportError:
        return "traci"


def epsilon_for_local_episode(local_episode):
    fraction = min(1.0, local_episode / EPSILON_DECAY_EPISODES)
    return EPSILON_START + fraction * (EPSILON_END - EPSILON_START)


def sample_scenario(rng):
    names = list(TRAIN_SCENARIO_WEIGHTS)
    weights = [TRAIN_SCENARIO_WEIGHTS[name] for name in names]
    return rng.choices(names, weights=weights, k=1)[0]


def run_training_episode(env, agent, buffer, epsilon, episode_seed, global_step):
    obs, info = env.reset(seed=episode_seed)
    metrics = EpisodeMetrics()
    metrics.record_step(
        waiting_time_s=info["total_waiting_time_s"],
        queue_length=int(obs.sum()),
        sim_time_s=info["sim_time_s"],
    )

    episode_reward = 0.0
    loss_sum = 0.0
    loss_count = 0
    arrived_vehicles = info["arrived_vehicles"]
    terminated = truncated = False

    while not (terminated or truncated):
        action = agent.act(obs, epsilon)
        next_obs, reward, terminated, truncated, info = env.step(action)
        buffer.push(Transition(obs, action, reward, next_obs, terminated))
        obs = next_obs
        episode_reward += reward
        arrived_vehicles = info["arrived_vehicles"]
        metrics.record_step(
            waiting_time_s=info["total_waiting_time_s"],
            queue_length=int(obs.sum()),
            sim_time_s=info["sim_time_s"],
        )
        global_step += 1

        if len(buffer) >= MIN_REPLAY_SIZE and global_step % TRAIN_EVERY_N_STEPS == 0:
            loss_sum += agent.train_step(buffer.sample(BATCH_SIZE))
            loss_count += 1

    final_metrics = metrics.finalize(arrived_vehicles=arrived_vehicles).to_dict()
    avg_loss = loss_sum / loss_count if loss_count else 0.0
    return final_metrics, episode_reward, avg_loss, global_step


rng = random.Random(RANDOM_SEED)
backend = resolve_backend()
agent = DQNAgent(learning_rate=LEARNING_RATE, gamma=GAMMA, seed=RANDOM_SEED)
base_episode = load_checkpoint(BASE_CHECKPOINT, agent)
for param_group in agent.optimizer.param_groups:
    param_group["lr"] = LEARNING_RATE
agent.sync_target_network()

buffer = ReplayBuffer(capacity=REPLAY_CAPACITY, seed=RANDOM_SEED)
global_step = 0
scenario_history = defaultdict(lambda: deque(maxlen=20))

print("Base checkpoint:", BASE_CHECKPOINT, "episode", base_episode)
print("Backend:", backend)
print("Training scenario weights:", TRAIN_SCENARIO_WEIGHTS)
print("Training scenario durations:", TRAIN_SCENARIO_DURATIONS)

progress = tqdm(range(1, FINETUNE_EPISODES + 1), desc="interleaved fine-tune", unit="ep")
for local_episode in progress:
    absolute_episode = base_episode + local_episode
    scenario_name = sample_scenario(rng)
    epsilon = epsilon_for_local_episode(local_episode)
    env = SumoTrafficEnv(
        sumocfg_path=SCENARIOS[scenario_name],
        episode_duration_s=TRAIN_SCENARIO_DURATIONS[scenario_name],
        green_duration_s=10.0,
        backend=backend,
        **GUARD_CONFIG,
    )
    try:
        metrics, episode_reward, avg_loss, global_step = run_training_episode(
            env,
            agent,
            buffer,
            epsilon,
            episode_seed=RANDOM_SEED + absolute_episode,
            global_step=global_step,
        )
    finally:
        env.close()

    scenario_history[scenario_name].append(metrics["mean_waiting_time_s"])

    if local_episode % TARGET_SYNC_EVERY_EPISODES == 0:
        agent.sync_target_network()

    if local_episode % CHECKPOINT_EVERY_EPISODES == 0 or local_episode == FINETUNE_EPISODES:
        save_checkpoint(CHECKPOINT_DIR / f"dqn_episode_{absolute_episode}.pt", agent, absolute_episode)

    progress.set_postfix(
        scenario=scenario_name,
        eps=f"{epsilon:.2f}",
        reward=f"{episode_reward:.0f}",
        wait=f"{metrics['mean_waiting_time_s']:.0f}",
        loss=f"{avg_loss:.2f}",
        replay=len(buffer),
    )

save_checkpoint(CHECKPOINT_DIR / "dqn_final.pt", agent, base_episode + FINETUNE_EPISODES)
print("Saved final checkpoint:", CHECKPOINT_DIR / "dqn_final.pt")
print("Recent mean wait by scenario:")
for scenario_name, waits in sorted(scenario_history.items()):
    print(f"{scenario_name:<22}", f"{sum(waits) / len(waits):8.2f}", "n=", len(waits))


## 7. Evaluate with hard guardrails and select `dqn_ew_repair_best.pt`

Evaluation dùng các scenario chuẩn, không dùng toàn bộ biến thể training. Một checkpoint chỉ được promote khi vượt qua cả aggregate guardrail và seed-level guardrail.

Điểm quan trọng: nếu không có checkpoint nào pass, notebook sẽ copy lại `dqn_eval_best.pt` làm `dqn_ew_repair_best.pt`. Như vậy file output luôn tồn tại, nhưng không bao giờ promote model nguy hiểm.


In [ ]:
from pathlib import Path
import shutil

from benchmark.policies import DQNPolicy, FixedTimePolicy
from benchmark.run_episode import run_episode
from rl.agent.dqn_agent import DQNAgent
from rl.env.traffic_env import SumoTrafficEnv
from rl.train.checkpoint import load_checkpoint

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
QUICK_EVAL_SEEDS = [101, 102]
FULL_EVAL_SEEDS = [101, 102, 103, 104, 105]
EVAL_SCENARIOS = ["normal", "heavy_2x", "imbalanced_ew3x", "imbalanced_ns3x", "mixed_balanced_peak"]
QUICK_EVAL_DURATIONS = {
    "normal": 900,
    "heavy_2x": 1200,
    "imbalanced_ew3x": 1200,
    "imbalanced_ns3x": 1200,
    "mixed_balanced_peak": 1600,
}

periodic = sorted(CHECKPOINT_DIR.glob("dqn_episode_*.pt"), key=lambda p: int(p.stem.split("_")[-1]))
quick_candidate_paths = []
for path in [CHECKPOINT_DIR / "dqn_eval_best.pt", CHECKPOINT_DIR / "dqn_best.pt", CHECKPOINT_DIR / "dqn_final.pt", *periodic[-8:]]:
    if path.exists() and path not in quick_candidate_paths:
        quick_candidate_paths.append(path)


def mean(values):
    return sum(values) / len(values) if values else 0.0


def aggregate(metrics):
    dicts = [m.to_dict() for m in metrics]
    return {key: mean([d[key] for d in dicts]) for key in dicts[0]}


def eval_policy(scenario_name, policy, seed, *, duration_s):
    env = SumoTrafficEnv(
        sumocfg_path=SCENARIOS[scenario_name],
        episode_duration_s=duration_s,
        green_duration_s=10.0,
        backend="libsumo",
        **GUARD_CONFIG,
    )
    try:
        return run_episode(env, policy, seed=seed)
    finally:
        env.close()


def improvement_pct(fixed, dqn):
    return {
        "mean_wait": (fixed["mean_waiting_time_s"] - dqn["mean_waiting_time_s"]) / fixed["mean_waiting_time_s"] * 100,
        "final_wait": (fixed["final_waiting_time_s"] - dqn["final_waiting_time_s"]) / fixed["final_waiting_time_s"] * 100 if fixed["final_waiting_time_s"] else 0.0,
        "queue": (fixed["mean_queue_length"] - dqn["mean_queue_length"]) / fixed["mean_queue_length"] * 100,
        "arrived": (dqn["arrived_vehicles"] - fixed["arrived_vehicles"]) / fixed["arrived_vehicles"] * 100,
    }


def per_seed_improvements(fixed_runs, dqn_runs):
    return [
        improvement_pct(fixed_metric.to_dict(), dqn_metric.to_dict())
        for fixed_metric, dqn_metric in zip(fixed_runs, dqn_runs)
    ]


def hard_guardrail_failures(agg_improvements, per_seed, details, seeds):
    failures = []

    if agg_improvements["normal"]["mean_wait"] < 5.0:
        failures.append("normal_mean_wait_regressed")
    if agg_improvements["normal"]["queue"] < 5.0:
        failures.append("normal_queue_regressed")
    if agg_improvements["heavy_2x"]["mean_wait"] < 5.0:
        failures.append("heavy_mean_wait_regressed")
    if agg_improvements["heavy_2x"]["arrived"] < -6.0:
        failures.append("heavy_throughput_drop")
    if agg_improvements["imbalanced_ns3x"]["mean_wait"] < 10.0:
        failures.append("ns3x_mean_wait_regressed")
    if agg_improvements["imbalanced_ns3x"]["arrived"] < 0.0:
        failures.append("ns3x_throughput_drop")
    if agg_improvements["mixed_balanced_peak"]["mean_wait"] < 60.0:
        failures.append("mixed_mean_wait_regressed")
    if agg_improvements["mixed_balanced_peak"]["arrived"] < 0.0:
        failures.append("mixed_throughput_drop")

    max_mean_wait_by_scenario = {
        "normal": 160.0,
        "heavy_2x": 900.0,
        "imbalanced_ew3x": 700.0,
        "imbalanced_ns3x": 700.0,
        "mixed_balanced_peak": 800.0,
    }
    max_final_wait_by_scenario = {
        "normal": 500.0,
        "heavy_2x": 2500.0,
        "imbalanced_ew3x": 2500.0,
        "imbalanced_ns3x": 2500.0,
        "mixed_balanced_peak": 3000.0,
    }

    for scenario_name, runs in details["per_seed_dqn"].items():
        for seed, metrics in zip(seeds, runs):
            if metrics["mean_waiting_time_s"] > max_mean_wait_by_scenario[scenario_name]:
                failures.append(f"{scenario_name}_seed_{seed}_mean_wait_spike")
            if metrics["final_waiting_time_s"] > max_final_wait_by_scenario[scenario_name]:
                failures.append(f"{scenario_name}_seed_{seed}_final_wait_spike")
            if metrics["arrived_vehicles"] <= 0:
                failures.append(f"{scenario_name}_seed_{seed}_no_throughput")

    for scenario_name, improvements in per_seed.items():
        for seed, imp in zip(seeds, improvements):
            if scenario_name == "normal" and imp["mean_wait"] < -10.0:
                failures.append(f"{scenario_name}_seed_{seed}_mean_wait_regressed")
            if scenario_name == "heavy_2x" and imp["arrived"] < -12.0:
                failures.append(f"{scenario_name}_seed_{seed}_throughput_regressed")
            if scenario_name in {"imbalanced_ew3x", "imbalanced_ns3x"} and imp["mean_wait"] < -25.0:
                failures.append(f"{scenario_name}_seed_{seed}_mean_wait_regressed")
            if scenario_name == "mixed_balanced_peak" and imp["arrived"] < -8.0:
                failures.append(f"{scenario_name}_seed_{seed}_throughput_regressed")

    if failures:
        return failures

    # Selection preference: after safety passes, choose controllers that improve
    # the weak EW peak while staying strong on the other production scenarios.
    ew = agg_improvements["imbalanced_ew3x"]
    if ew["mean_wait"] < 20.0:
        failures.append("ew3x_mean_wait_not_enough")
    if ew["queue"] < 5.0:
        failures.append("ew3x_queue_not_enough")
    if ew["arrived"] < 0.0:
        failures.append("ew3x_throughput_drop")

    return failures


def policy_from_checkpoint(path):
    candidate_agent = DQNAgent(learning_rate=LEARNING_RATE, gamma=GAMMA, seed=RANDOM_SEED)
    load_checkpoint(path, candidate_agent)
    candidate_agent.epsilon = 0.0
    return DQNPolicy(candidate_agent, epsilon=0.0)


def evaluate_candidates(candidate_paths, seeds, durations, *, label):
    print(f"{label}: evaluating {len(candidate_paths)} candidates x {len(EVAL_SCENARIOS)} scenarios x {len(seeds)} seeds")
    fixed_cache = {}
    for scenario_name in EVAL_SCENARIOS:
        fixed_cache[scenario_name] = [
            eval_policy(scenario_name, FixedTimePolicy(green_duration_s=20.0), seed, duration_s=durations[scenario_name])
            for seed in seeds
        ]
        print(f"Fixed-time {label} baseline ready:", scenario_name)

    results = []
    for path in candidate_paths:
        policy = policy_from_checkpoint(path)
        scenario_improvements = {}
        scenario_metrics = {}
        per_seed = {}
        per_seed_dqn = {}

        for scenario_name in EVAL_SCENARIOS:
            dqn_runs = [
                eval_policy(scenario_name, policy, seed, duration_s=durations[scenario_name])
                for seed in seeds
            ]
            fixed_agg = aggregate(fixed_cache[scenario_name])
            dqn_agg = aggregate(dqn_runs)
            scenario_improvements[scenario_name] = improvement_pct(fixed_agg, dqn_agg)
            scenario_metrics[scenario_name] = dqn_agg
            per_seed[scenario_name] = per_seed_improvements(fixed_cache[scenario_name], dqn_runs)
            per_seed_dqn[scenario_name] = [m.to_dict() for m in dqn_runs]

        score = (
            1.60 * scenario_improvements["imbalanced_ew3x"]["mean_wait"]
            + 1.10 * scenario_improvements["imbalanced_ns3x"]["mean_wait"]
            + 0.90 * scenario_improvements["heavy_2x"]["mean_wait"]
            + 0.80 * scenario_improvements["normal"]["mean_wait"]
            + 0.80 * scenario_improvements["mixed_balanced_peak"]["mean_wait"]
            + 0.25 * scenario_improvements["imbalanced_ew3x"]["arrived"]
        )
        results.append(
            {
                "path": path,
                "score": score,
                "improvements": scenario_improvements,
                "metrics": scenario_metrics,
                "per_seed": per_seed,
                "details": {"per_seed_dqn": per_seed_dqn},
            }
        )
        print(label, path.name, "score", round(score, 2), scenario_improvements)

    return sorted(results, key=lambda item: item["score"], reverse=True)


quick_results = evaluate_candidates(quick_candidate_paths, QUICK_EVAL_SEEDS, QUICK_EVAL_DURATIONS, label="quick")
top_repair_candidates = [result["path"] for result in quick_results if result["path"].name != "dqn_eval_best.pt"][:3]
full_candidate_paths = []
for path in [CHECKPOINT_DIR / "dqn_eval_best.pt", *top_repair_candidates]:
    if path.exists() and path not in full_candidate_paths:
        full_candidate_paths.append(path)

full_results = evaluate_candidates(full_candidate_paths, FULL_EVAL_SEEDS, SCENARIO_DURATIONS, label="full")
passed = []
for result in full_results:
    failures = hard_guardrail_failures(result["improvements"], result["per_seed"], result["details"], FULL_EVAL_SEEDS)
    result["failures"] = failures
    print("FULL", result["path"].name, "score", round(result["score"], 2), "failures", failures)
    if result["path"].name != "dqn_eval_best.pt" and not failures:
        passed.append(result)

if passed:
    best = max(passed, key=lambda item: item["score"])
    shutil.copy2(best["path"], CHECKPOINT_DIR / "dqn_ew_repair_best.pt")
    print("Selected repair checkpoint:", best["path"], "score", best["score"])
else:
    shutil.copy2(CHECKPOINT_DIR / "dqn_eval_best.pt", CHECKPOINT_DIR / "dqn_ew_repair_best.pt")
    print("No repair candidate passed hard guardrails; kept safe base checkpoint.")


## 8. Download fine-tuned checkpoints

In [ ]:
!ls -lh /kaggle/working/checkpoints

import base64
from pathlib import Path
from IPython.display import HTML, display


def download_link(path, filename=None):
    filename = filename or path.split("/")[-1]
    with open(path, "rb") as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    return HTML(f'<a download="{filename}" href="data:application/octet-stream;base64,{b64}">Download {filename}</a>')


for path in [
    "/kaggle/working/checkpoints/dqn_ew_repair_best.pt",
    "/kaggle/working/checkpoints/dqn_eval_best.pt",
    "/kaggle/working/checkpoints/dqn_best.pt",
    "/kaggle/working/checkpoints/dqn_final.pt",
]:
    if Path(path).exists():
        display(download_link(path))
